In [1]:
import json

with open('../data/GraphDTA/data/davis/ligands_can.txt') as f:
    ligands = json.load(f)

print('Total number of drugs in Davis:', len(ligands))

first_id = list(ligands.keys())[0]
first_smiles = ligands[first_id]

print('Drug ID:', first_id)
print('SMILES :', first_smiles)

Total number of drugs in Davis: 68
Drug ID: 11314340
SMILES : CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N


In [2]:
from rdkit import Chem

mol = Chem.MolFromSmiles(first_smiles)

print('Parsed OK:', mol is not None)
print('Number of atoms (heavy atoms, no H):', mol.GetNumAtoms())
print('Number of bonds:', mol.GetNumBonds())

Parsed OK: True
Number of atoms (heavy atoms, no H): 27
Number of bonds: 30


In [3]:
for atom in mol.GetAtoms():
    print(
        f"Atom idx {atom.GetIdx():2d} | "
        f"element={atom.GetSymbol():2s} | "
        f"degree={atom.GetDegree()} | "
        f"aromatic={atom.GetIsAromatic()} | "
        f"in_ring={atom.IsInRing()} | "
        f"charge={atom.GetFormalCharge()}"
    )

Atom idx  0 | element=C  | degree=1 | aromatic=False | in_ring=False | charge=0
Atom idx  1 | element=C  | degree=3 | aromatic=True | in_ring=True | charge=0
Atom idx  2 | element=C  | degree=3 | aromatic=True | in_ring=True | charge=0
Atom idx  3 | element=C  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx  4 | element=C  | degree=3 | aromatic=True | in_ring=True | charge=0
Atom idx  5 | element=C  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx  6 | element=C  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx  7 | element=C  | degree=3 | aromatic=True | in_ring=True | charge=0
Atom idx  8 | element=N  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx  9 | element=N  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx 10 | element=C  | degree=3 | aromatic=True | in_ring=True | charge=0
Atom idx 11 | element=C  | degree=2 | aromatic=True | in_ring=True | charge=0
Atom idx 12 | element=C  | degree=3 | aromatic=True | in_ring=

In [5]:
for bond in mol.GetBonds():
    print(
        f"Bond: atom {bond.GetBeginAtomIdx():2d} -- atom {bond.GetEndAtomIdx():2d} "
        f"| type={bond.GetBondTypeAsDouble()} | aromatic={bond.GetIsAromatic()}"
    )

Bond: atom  0 -- atom  1 | type=1.0 | aromatic=False
Bond: atom  1 -- atom  2 | type=1.5 | aromatic=True
Bond: atom  2 -- atom  3 | type=1.5 | aromatic=True
Bond: atom  3 -- atom  4 | type=1.5 | aromatic=True
Bond: atom  4 -- atom  5 | type=1.5 | aromatic=True
Bond: atom  5 -- atom  6 | type=1.5 | aromatic=True
Bond: atom  6 -- atom  7 | type=1.5 | aromatic=True
Bond: atom  7 -- atom  8 | type=1.5 | aromatic=True
Bond: atom  8 -- atom  9 | type=1.5 | aromatic=True
Bond: atom  4 -- atom 10 | type=1.0 | aromatic=False
Bond: atom 10 -- atom 11 | type=1.5 | aromatic=True
Bond: atom 11 -- atom 12 | type=1.5 | aromatic=True
Bond: atom 12 -- atom 13 | type=1.5 | aromatic=True
Bond: atom 13 -- atom 14 | type=1.5 | aromatic=True
Bond: atom 14 -- atom 15 | type=1.5 | aromatic=True
Bond: atom 12 -- atom 16 | type=1.0 | aromatic=False
Bond: atom 16 -- atom 17 | type=1.0 | aromatic=False
Bond: atom 17 -- atom 18 | type=1.0 | aromatic=False
Bond: atom 18 -- atom 19 | type=1.0 | aromatic=False
Bond: 

In [6]:
ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

first_atom = mol.GetAtomWithIdx(0)
print('First atom element:', first_atom.GetSymbol())
print('Its feature vector:', atom_features(first_atom))
print('Vector length:', len(atom_features(first_atom)))

First atom element: C
Its feature vector: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Vector length: 14


In [7]:
print('ATOM_VOCAB length:', len(ATOM_VOCAB))
print('one_hot result for C:', one_hot('C', ATOM_VOCAB))
print('degree:', first_atom.GetDegree())
print('charge:', first_atom.GetFormalCharge())
print('aromatic:', int(first_atom.GetIsAromatic()))
print('in_ring:', int(first_atom.IsInRing()))

ATOM_VOCAB length: 9
one_hot result for C: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
degree: 1
charge: 0
aromatic: 0
in_ring: 0


In [8]:
all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]

print('Number of atoms:', len(all_atom_features))
print('Feature vector length per atom:', len(all_atom_features[0]))
print()
for i, feats in enumerate(all_atom_features):
    print(i, feats)

Number of atoms: 27
Feature vector length per atom: 14

0 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
1 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
2 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
3 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
4 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
5 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
6 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
7 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
8 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
9 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
10 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
11 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
12 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 1]
13 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
14 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
15 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 1]
16 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0]
17 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0]
18 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0]
19 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0]
20 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
edge_sources = []
edge_targets = []

for bond in mol.GetBonds():
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()
    edge_sources += [i, j]
    edge_targets += [j, i]

print('Number of directed edges (2x bonds):', len(edge_sources))
print('Source indices:', edge_sources)
print('Target indices:', edge_targets)

Number of directed edges (2x bonds): 60
Source indices: [0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 4, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 12, 16, 16, 17, 17, 18, 18, 19, 19, 20, 20, 21, 21, 22, 22, 23, 23, 24, 24, 25, 18, 26, 9, 1, 7, 2, 15, 10, 25, 20]
Target indices: [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 6, 8, 7, 9, 8, 10, 4, 11, 10, 12, 11, 13, 12, 14, 13, 15, 14, 16, 12, 17, 16, 18, 17, 19, 18, 20, 19, 21, 20, 22, 21, 23, 22, 24, 23, 25, 24, 26, 18, 1, 9, 2, 7, 10, 15, 20, 25]


In [10]:
import torch
from torch_geometric.data import Data

x = torch.tensor(all_atom_features, dtype=torch.float)
edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)

graph = Data(x=x, edge_index=edge_index)

print(graph)
print()
print('x shape (num_atoms, num_features):', graph.x.shape)
print('edge_index shape (2, num_directed_edges):', graph.edge_index.shape)

Data(x=[27, 14], edge_index=[2, 60])

x shape (num_atoms, num_features): torch.Size([27, 14])
edge_index shape (2, num_directed_edges): torch.Size([2, 60])


In [11]:
from torch_geometric.nn import GCNConv

conv_layer = GCNConv(in_channels=14, out_channels=8)

updated_x = conv_layer(graph.x, graph.edge_index)

print('Before layer, x shape:', graph.x.shape)
print('After layer, x shape:', updated_x.shape)
print()
print('Atom 0 vector BEFORE:', graph.x[0].tolist())
print('Atom 0 vector AFTER :', updated_x[0].tolist())

Before layer, x shape: torch.Size([27, 14])
After layer, x shape: torch.Size([27, 8])

Atom 0 vector BEFORE: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
Atom 0 vector AFTER : [-0.4018663763999939, 0.023779459297657013, -0.26188451051712036, -1.282638669013977, -0.36491572856903076, -0.3834758698940277, -0.652988851070404, -1.316989779472351]


In [12]:
import torch.nn.functional as F

conv1 = GCNConv(in_channels=14, out_channels=16)
conv2 = GCNConv(in_channels=16, out_channels=16)
conv3 = GCNConv(in_channels=16, out_channels=16)

h = conv1(graph.x, graph.edge_index)
h = F.relu(h)
h = conv2(h, graph.edge_index)
h = F.relu(h)
h = conv3(h, graph.edge_index)

print('Final atom representations shape:', h.shape)
print()
print('Atom 0 final vector:', h[0].tolist())

Final atom representations shape: torch.Size([27, 16])

Atom 0 final vector: [0.24914544820785522, -0.4913966655731201, -0.4946615695953369, -0.02980867400765419, 0.3150023818016052, 0.1984332799911499, -0.47488272190093994, -0.23757144808769226, -0.20606295764446259, -0.20720511674880981, 0.0826476663351059, 0.000886841444298625, -0.5663684606552124, -0.6861649751663208, 0.174181267619133, 0.6285063028335571]


In [13]:
mean_pooled = h.mean(dim=0)

print('Shape of all atom vectors:', h.shape)
print('Shape after mean pooling:', mean_pooled.shape)
print()
print('Molecule vector (mean pooled):', mean_pooled.tolist())

Shape of all atom vectors: torch.Size([27, 16])
Shape after mean pooling: torch.Size([16])

Molecule vector (mean pooled): [0.3158099949359894, -0.632175087928772, -0.6474843621253967, -0.053895171731710434, 0.4210264980792999, 0.2602534890174866, -0.6077489256858826, -0.3044504225254059, -0.27859026193618774, -0.2777515649795532, 0.10214943438768387, 0.019932134076952934, -0.7325987815856934, -0.8827289342880249, 0.21734417974948883, 0.8227275013923645]


In [14]:
import torch.nn as nn

attention_scorer = nn.Linear(16, 1)

raw_scores = attention_scorer(h)
print('Raw scores shape:', raw_scores.shape)
print('Raw scores (one per atom):', raw_scores.squeeze().tolist())

attention_weights = F.softmax(raw_scores, dim=0)
print()
print('Attention weights (should sum to ~1.0):', attention_weights.squeeze().tolist())
print('Sum of weights:', attention_weights.sum().item())

attention_pooled = (attention_weights * h).sum(dim=0)
print()
print('Molecule vector (attention pooled):', attention_pooled.tolist())

Raw scores shape: torch.Size([27, 1])
Raw scores (one per atom): [0.25106024742126465, 0.3993208706378937, 0.41778677701950073, 0.35593029856681824, 0.4199976325035095, 0.34668394923210144, 0.3462568521499634, 0.4139764904975891, 0.3412367105484009, 0.33631396293640137, 0.410561203956604, 0.3369711637496948, 0.37880900502204895, 0.32216528058052063, 0.32700416445732117, 0.33859679102897644, 0.30224406719207764, 0.2988642454147339, 0.367848664522171, 0.3284584581851959, 0.3976764678955078, 0.33222144842147827, 0.32199007272720337, 0.32013481855392456, 0.32199007272720337, 0.33222144842147827, 0.2317093312740326]

Attention weights (should sum to ~1.0): [0.033702302724123, 0.03908844292163849, 0.039816949516534805, 0.03742864355444908, 0.03990507870912552, 0.03708415850996971, 0.03706832602620125, 0.039665527641773224, 0.03688270226120949, 0.03670158237218857, 0.03953028842806816, 0.036725711077451706, 0.03829482942819595, 0.036185961216688156, 0.036361485719680786, 0.03678546100854874, 